# Model Training & Hyperparameter Tuning
**COEN 330 - Applied Machine Learning**

Trains and tunes **five** classifiers on the loan-approval task. Read top to bottom:
one section per model, each showing baseline → tuning → per-fold cross-validation.

**Validation:** 5-fold StratifiedKFold on the **training set only**. The test set is loaded once at the top but is not used until final evaluation to prevent data leakage.

**Leakage-safe:** the preprocessor lives *inside* a `Pipeline`, so the scaler/encoder
refit on each fold's training portion during CV (never on the whole training set first).
Each saved model is a full pipeline (raw DataFrame in → prediction out).

**Target:** `loan_status` (1 = approved, 0 = rejected). Positive class = approved,
the **minority (~22%)**, so we use `class_weight='balanced'` where supported and
**select on PR-AUC**.

**Reported metrics:** accuracy, balanced accuracy, precision, recall, F1, PR-AUC.

Models: Logistic Regression (baseline), SVM (RBF), Gaussian Naive Bayes, Random Forest,
Gradient Boosting (HistGradientBoosting). A Dummy classifier is a floor check, not one of the five models.

## Setup

In [1]:
import sys, json, time
from pathlib import Path
sys.path.append(str(Path.cwd().parent / 'src'))   # make src/ importable

import numpy as np
import pandas as pd
import joblib
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from preprocessing import load_data, split_data, build_preprocessor, get_feature_names
from utils import DATA_RAW, DATASET_FILE, MODELS_DIR, RESULTS_DIR, SEED

DATA_PATH = DATA_RAW / DATASET_FILE

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Multi-metric scoring; we REFIT (select) on PR-AUC. (No ROC-AUC by choice.)
SCORING = {
    'pr_auc':       'average_precision',
    'accuracy':     'accuracy',
    'balanced_acc': 'balanced_accuracy',
    'precision':    'precision',
    'recall':       'recall',
    'f1':           'f1',
}
PRIMARY = 'pr_auc'
METRIC_ORDER = ['pr_auc', 'f1', 'precision', 'recall', 'accuracy', 'balanced_acc']

## Load and Split Traint/Test

`X_train` / `X_test` are raw DataFrames. The split is deterministic (fixed seed), so
the (04) evaluation notebook reproduces the identical split.

In [2]:
df = load_data(DATA_PATH, add_engineered=False)
X_train, X_test, y_train, y_test = split_data(df)

print('='*60)
print(f'TRAIN: {len(X_train):,} rows   |   approved (1) = {y_train.mean():.1%}')
print(f'TEST : {len(X_test):,} rows   <-- SET ASIDE, used ONLY in 04_evaluation')
print('='*60)

TRAIN: 36,000 rows   |   approved (1) = 22.2%
TEST : 9,000 rows   <-- SET ASIDE, used ONLY in 04_evaluation


## Shared helpers

Defined once so each model section below is short and readable:
- `make_pipe` wraps preprocessor + classifier (leakage-safe).
- `cv_eval` cross-validates a config and returns per-fold mean/std for all 6 metrics.
- `tune` runs GridSearchCV (refit on PR-AUC).
- `run_model` ties it together: baseline CV → tune → tuned per-fold CV → compare → store.

In [3]:
best_pipes = {}   # name -> fitted pipeline
rows = []          # comparison table rows

def make_pipe(estimator):
    return Pipeline([('pre', build_preprocessor()), ('clf', estimator)])

def _summarise(folds_by_metric):
    out = {}
    for m, folds in folds_by_metric.items():
        folds = np.asarray(folds, dtype=float)
        out[m] = {'mean': folds.mean(), 'std': folds.std(), 'folds': folds.tolist()}
    return out

def cv_eval(pipe, X, y):
    # Cross-validate one fixed config; return per-fold mean/std/folds per metric.
    res = cross_validate(pipe, X, y, cv=cv, scoring=SCORING, n_jobs=-1)
    return _summarise({m: res[f'test_{m}'] for m in SCORING})

def tune(estimator, grid, X, y):
    pipe = make_pipe(estimator)
    pgrid = {f'clf__{k}': v for k, v in grid.items()}   # target the classifier step
    gs = GridSearchCV(pipe, pgrid, scoring=SCORING, refit=PRIMARY, cv=cv, n_jobs=-1)
    gs.fit(X, y)
    return gs

def _per_fold_from_gs(gs):
    # Per-fold scores for the BEST config, pulled from cv_results_ (free, exact same folds).
    i = gs.best_index_
    n = cv.get_n_splits()
    return _summarise({m: [gs.cv_results_[f'split{k}_test_{m}'][i] for k in range(n)]
                       for m in SCORING})

def _strip(params):
    return {k.replace('clf__', ''): v for k, v in params.items()}

def _print_cv(title, summary, show_folds=False):
    print(f'  {title}')
    for m in METRIC_ORDER:
        s = summary[m]
        if show_folds:
            folds = ' '.join(f'{v:.3f}' for v in s['folds'])
            print(f'    {m:13s} mean={s["mean"]:.4f}  std={s["std"]:.4f}  folds=[{folds}]')
        else:
            print(f'    {m:13s} mean={s["mean"]:.4f}  std={s["std"]:.4f}')

def run_model(name, estimator, grid):
    print('='*60); print(name); print('='*60)
    t0 = time.time()

    # 1) Baseline (estimator as-is, no tuning) -> CV
    base = cv_eval(make_pipe(clone(estimator)), X_train, y_train)
    _print_cv('Baseline  — 5-fold CV (means/std):', base, show_folds=False)

    # 2) Tune
    gs = tune(estimator, grid, X_train, y_train)
    print(f'  Best params: {_strip(gs.best_params_)}')

    # 3) Tuned config -> per-fold variance (from the search, same folds)
    tuned = _per_fold_from_gs(gs)
    _print_cv('Tuned     — 5-fold CV (per fold):', tuned, show_folds=True)

    # 4) Baseline vs tuned (headline deltas)
    for m in ['pr_auc', 'f1']:
        b, t = base[m]['mean'], tuned[m]['mean']
        print(f'  {m:6s}: baseline={b:.4f} -> tuned={t:.4f}  (delta {t-b:+.4f})')
    print(f'  ({time.time()-t0:.1f}s)')

    # 5) Store: full fitted pipeline + a comparison row (tuned means + stds)
    best_pipes[name] = gs.best_estimator_
    row = {'model': name, 'best_params': _strip(gs.best_params_)}
    row.update({m: tuned[m]['mean'] for m in SCORING})
    row.update({f'{m}_std': tuned[m]['std'] for m in SCORING})
    rows.append(row)
    return gs

## Dummy baseline (floor)

Predict the majority class (rejected). High accuracy here is exactly why we do **not**
select on accuracy.

In [4]:
from sklearn.metrics import average_precision_score, recall_score
dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
dp = dummy.predict_proba(X_test)[:, 1]
print(f'Dummy accuracy (test) : {dummy.score(X_test, y_test):.3f}')
print(f'Dummy PR-AUC (test)   : {average_precision_score(y_test, dp):.3f}  (= base rate {y_test.mean():.3f})')
print(f'Dummy recall on approved: {recall_score(y_test, dummy.predict(X_test)):.3f}  (predicts no approvals)')

Dummy accuracy (test) : 0.778
Dummy PR-AUC (test)   : 0.222  (= base rate 0.222)
Dummy recall on approved: 0.000  (predicts no approvals)


## 1. Logistic Regression  (interpretable baseline)

Elastic-net via `saga` (`l1_ratio=0` → L2, `1` → L1). `class_weight='balanced'` for the
minority class.

In [5]:
run_model(
    'LogisticRegression',
    LogisticRegression(solver='saga', penalty='elasticnet', l1_ratio=0.5,
                       class_weight='balanced', max_iter=2000, random_state=SEED),
    {'C': [0.01, 0.1, 1, 10], 'l1_ratio': [0, 1]},
);

LogisticRegression
  Baseline  — 5-fold CV (means/std):
    pr_auc        mean=0.8558  std=0.0036
    f1            mean=0.7440  std=0.0029
    precision     mean=0.6226  std=0.0029
    recall        mean=0.9241  std=0.0076
    accuracy      mean=0.8587  std=0.0015
    balanced_acc  mean=0.8820  std=0.0031


d:\Concordia Courses\Summer 2026\COEN 330\COEN 330 Project\COEN330-Machine-Learning-Project\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


  Best params: {'C': 0.01, 'l1_ratio': 0}
  Tuned     — 5-fold CV (per fold):
    pr_auc        mean=0.8562  std=0.0040  folds=[0.852 0.851 0.859 0.861 0.859]
    f1            mean=0.7415  std=0.0027  folds=[0.746 0.737 0.741 0.741 0.743]
    precision     mean=0.6189  std=0.0035  folds=[0.624 0.618 0.619 0.613 0.620]
    recall        mean=0.9246  std=0.0074  folds=[0.926 0.913 0.923 0.936 0.925]
    accuracy      mean=0.8567  std=0.0018  folds=[0.860 0.855 0.857 0.855 0.858]
    balanced_acc  mean=0.8810  std=0.0028  folds=[0.883 0.876 0.880 0.884 0.882]
  pr_auc: baseline=0.8558 -> tuned=0.8562  (delta +0.0003)
  f1    : baseline=0.7440 -> tuned=0.7415  (delta -0.0025)
  (65.1s)


## 2. SVM (RBF kernel)

Margin-based. Slowest model on this many rows, so the grid is small. Uses
`decision_function` for PR-AUC (no `probability=True` needed → faster).

In [6]:
run_model(
    'SVM_RBF',
    SVC(kernel='rbf', class_weight='balanced', random_state=SEED),
    {'C': [1, 10], 'gamma': ['scale']},
);

SVM_RBF
  Baseline  — 5-fold CV (means/std):
    pr_auc        mean=0.8863  std=0.0080
    f1            mean=0.7656  std=0.0045
    precision     mean=0.6517  std=0.0040
    recall        mean=0.9279  std=0.0104
    accuracy      mean=0.8738  std=0.0022
    balanced_acc  mean=0.8931  std=0.0046
  Best params: {'C': 1, 'gamma': 'scale'}
  Tuned     — 5-fold CV (per fold):
    pr_auc        mean=0.8863  std=0.0080  folds=[0.882 0.873 0.891 0.890 0.896]
    f1            mean=0.7656  std=0.0045  folds=[0.770 0.758 0.764 0.767 0.770]
    precision     mean=0.6517  std=0.0040  folds=[0.659 0.648 0.649 0.651 0.650]
    recall        mean=0.9279  std=0.0104  folds=[0.924 0.912 0.927 0.932 0.944]
    accuracy      mean=0.8738  std=0.0022  folds=[0.877 0.871 0.873 0.874 0.875]
    balanced_acc  mean=0.8931  std=0.0046  folds=[0.894 0.885 0.892 0.895 0.899]
  pr_auc: baseline=0.8863 -> tuned=0.8863  (delta +0.0000)
  f1    : baseline=0.7656 -> tuned=0.7656  (delta +0.0000)
  (156.8s)


## 3. Gaussian Naive Bayes (probabilistic)

The independence assumption is broken by the one-hot columns, so expect it to trail —
a clean error-analysis talking point. Barely tunes.

In [7]:
run_model(
    'GaussianNB',
    GaussianNB(),
    {'var_smoothing': [1e-9, 1e-8, 1e-7]},
);

GaussianNB
  Baseline  — 5-fold CV (means/std):
    pr_auc        mean=0.8210  std=0.0040
    f1            mean=0.6232  std=0.0027
    precision     mean=0.4526  std=0.0028
    recall        mean=1.0000  std=0.0000
    accuracy      mean=0.7312  std=0.0031
    balanced_acc  mean=0.8272  std=0.0020
  Best params: {'var_smoothing': 1e-09}
  Tuned     — 5-fold CV (per fold):
    pr_auc        mean=0.8210  std=0.0040  folds=[0.821 0.816 0.821 0.820 0.828]
    f1            mean=0.6232  std=0.0027  folds=[0.625 0.626 0.623 0.618 0.623]
    precision     mean=0.4526  std=0.0028  folds=[0.455 0.456 0.452 0.448 0.452]
    recall        mean=1.0000  std=0.0000  folds=[1.000 1.000 1.000 1.000 1.000]
    accuracy      mean=0.7312  std=0.0031  folds=[0.733 0.735 0.731 0.726 0.731]
    balanced_acc  mean=0.8272  std=0.0020  folds=[0.829 0.829 0.827 0.824 0.827]
  pr_auc: baseline=0.8210 -> tuned=0.8210  (delta +0.0000)
  f1    : baseline=0.6232 -> tuned=0.6232  (delta +0.0000)
  (0.9s)


## 4. Random Forest (bagging ensemble)

In [9]:
run_model(
    'RandomForest',
    RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1),
    {'n_estimators': [200, 400], 'max_depth': [None, 10, 20]},
);

RandomForest
  Baseline  — 5-fold CV (means/std):
    pr_auc        mean=0.9220  std=0.0059
    f1            mean=0.8228  std=0.0098
    precision     mean=0.8022  std=0.0079
    recall        mean=0.8446  std=0.0126
    accuracy      mean=0.9192  std=0.0042
    balanced_acc  mean=0.8926  std=0.0072
  Best params: {'max_depth': 20, 'n_estimators': 400}
  Tuned     — 5-fold CV (per fold):
    pr_auc        mean=0.9261  std=0.0048  folds=[0.924 0.918 0.932 0.927 0.930]
    f1            mean=0.8233  std=0.0095  folds=[0.823 0.810 0.836 0.817 0.831]
    precision     mean=0.7891  std=0.0071  folds=[0.791 0.776 0.797 0.792 0.789]
    recall        mean=0.8606  std=0.0149  folds=[0.858 0.846 0.878 0.843 0.877]
    accuracy      mean=0.9179  std=0.0041  folds=[0.918 0.912 0.923 0.916 0.921]
    balanced_acc  mean=0.8975  std=0.0077  folds=[0.897 0.888 0.907 0.890 0.905]
  pr_auc: baseline=0.9220 -> tuned=0.9261  (delta +0.0041)
  f1    : baseline=0.8228 -> tuned=0.8233  (delta +0.0004)
  (5

## 5. Gradient Boosting (HistGradientBoosting — boosting ensemble)

No `class_weight`; imbalance is handled via PR-AUC selection here and threshold tuning in `04`.

In [10]:
run_model(
    'GradientBoosting',
    HistGradientBoostingClassifier(random_state=SEED),
    {'learning_rate': [0.05, 0.1], 'max_depth': [None, 6], 'max_iter': [200, 400]},
);

GradientBoosting
  Baseline  — 5-fold CV (means/std):
    pr_auc        mean=0.9347  std=0.0041
    f1            mean=0.8327  std=0.0096
    precision     mean=0.8877  std=0.0086
    recall        mean=0.7842  std=0.0142
    accuracy      mean=0.9300  std=0.0037
    balanced_acc  mean=0.8779  std=0.0072
  Best params: {'learning_rate': 0.1, 'max_depth': 6, 'max_iter': 200}
  Tuned     — 5-fold CV (per fold):
    pr_auc        mean=0.9365  std=0.0040  folds=[0.935 0.929 0.940 0.938 0.940]
    f1            mean=0.8370  std=0.0106  folds=[0.828 0.823 0.848 0.835 0.850]
    precision     mean=0.8933  std=0.0103  folds=[0.883 0.889 0.887 0.894 0.913]
    recall        mean=0.7875  std=0.0154  folds=[0.780 0.766 0.812 0.784 0.796]
    accuracy      mean=0.9319  std=0.0041  folds=[0.928 0.927 0.935 0.931 0.938]
    balanced_acc  mean=0.8803  std=0.0079  folds=[0.875 0.869 0.891 0.879 0.887]
  pr_auc: baseline=0.9347 -> tuned=0.9365  (delta +0.0018)
  f1    : baseline=0.8327 -> tuned=0.8370 

## 6. Comparison table (tuned models, sorted by PR-AUC)

In [13]:
results = (pd.DataFrame(rows)
           .sort_values('pr_auc', ascending=False)
           .reset_index(drop=True))

# Show means with their fold-to-fold std (stability) next to each metric.
view = results[['model'] + list(SCORING)].copy()
for m in SCORING:
    view[m] = results.apply(lambda r: f"{r[m]:.3f} (+/-{r[f'{m}_std']:.3f})", axis=1)
print('Tuned models — 5-fold CV (mean +/- std):')
display(view)
print('\nTop model by PR-AUC:', results.loc[0, 'model'])

Tuned models — 5-fold CV (mean +/- std):


,model,pr_auc,accuracy,balanced_acc,precision,recall,f1
0,GradientBoosting,0.937 (+/-0.004),0.932 (+/-0.004),0.880 (+/-0.008),0.893 (+/-0.010),0.787 (+/-0.015),0.837 (+/-0.011)
1,RandomForest,0.926 (+/-0.005),0.918 (+/-0.004),0.897 (+/-0.008),0.789 (+/-0.007),0.861 (+/-0.015),0.823 (+/-0.009)
2,RandomForest,0.926 (+/-0.005),0.918 (+/-0.004),0.897 (+/-0.008),0.789 (+/-0.007),0.861 (+/-0.015),0.823 (+/-0.009)
3,SVM_RBF,0.886 (+/-0.008),0.874 (+/-0.002),0.893 (+/-0.005),0.652 (+/-0.004),0.928 (+/-0.010),0.766 (+/-0.004)
4,LogisticRegression,0.856 (+/-0.004),0.857 (+/-0.002),0.881 (+/-0.003),0.619 (+/-0.004),0.925 (+/-0.007),0.741 (+/-0.003)
5,GaussianNB,0.821 (+/-0.004),0.731 (+/-0.003),0.827 (+/-0.002),0.453 (+/-0.003),1.000 (+/-0.000),0.623 (+/-0.003)



Top model by PR-AUC: GradientBoosting


## 7. Save tuned pipelines and CV results

In [14]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for name, pipe in best_pipes.items():
    joblib.dump(pipe, MODELS_DIR / f'{name}.pkl')

fitted_pre = best_pipes[next(iter(best_pipes))].named_steps['pre']
feature_names = get_feature_names(fitted_pre)
with open(MODELS_DIR / 'feature_names.json', 'w') as f:
    json.dump(feature_names, f)

results.to_csv(RESULTS_DIR / 'cv_results_primary.csv', index=False)
print(f'Saved {len(best_pipes)} tuned pipelines to {MODELS_DIR}')
print(f'{len(feature_names)} encoded features recorded')
print(f'Saved CV results to {RESULTS_DIR / "cv_results_primary.csv"}')

Saved 5 tuned pipelines to D:\Concordia Courses\Summer 2026\COEN 330\COEN 330 Project\COEN330-Machine-Learning-Project\models
23 encoded features recorded
Saved CV results to D:\Concordia Courses\Summer 2026\COEN 330\COEN 330 Project\COEN330-Machine-Learning-Project\results\cv_results_primary.csv


## 8. (Optional) Feature-set experiments

Re-run on alternative feature sets and compare CV PR-AUC: WITHOUT
`previous_loan_defaults_on_file`, and WITH the engineered ratio features. Left as a
function so it stays out of the main demo path; uncomment to run (slow).

In [ ]:
def run_feature_set(drop_prev_defaults, add_engineered, tag):
    df_ = load_data(DATA_PATH, add_engineered=add_engineered)
    Xtr, _, ytr, _ = split_data(df_)
    def mp(est):
        return Pipeline([('pre', build_preprocessor(drop_prev_defaults=drop_prev_defaults,
                                                    add_engineered=add_engineered)), ('clf', est)])
    specs = {
        'LogisticRegression': (LogisticRegression(solver='saga', penalty='elasticnet', l1_ratio=0.5,
                               class_weight='balanced', max_iter=2000, random_state=SEED),
                               {'C':[0.01,0.1,1,10],'l1_ratio':[0,1]}),
        'RandomForest': (RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1),
                         {'n_estimators':[200,400],'max_depth':[None,10,20]}),
        'GradientBoosting': (HistGradientBoostingClassifier(random_state=SEED),
                             {'learning_rate':[0.05,0.1],'max_depth':[None,6],'max_iter':[200,400]}),
    }
    out = []
    for name,(est,grid) in specs.items():
        gs = GridSearchCV(mp(est), {f'clf__{k}':v for k,v in grid.items()},
                          scoring=SCORING, refit=PRIMARY, cv=cv, n_jobs=-1).fit(Xtr, ytr)
        out.append({'feature_set':tag,'model':name,'pr_auc':gs.best_score_})
    return pd.DataFrame(out)

# arms = [run_feature_set(False, False, 'WITH_prevdef'),
#         run_feature_set(True,  False, 'WITHOUT_prevdef'),
#         run_feature_set(False, True,  'WITH_engineered')]
# comparison = pd.concat(arms, ignore_index=True)
# comparison.to_csv(RESULTS_DIR / 'feature_set_comparison.csv', index=False)
# display(comparison.pivot_table(index='model', columns='feature_set', values='pr_auc').round(4))